In [ ]:
df_config = spark.sql('SELECT * from risam_demo1.opendoor.source_to_s3_dataload_config')

* adding this

In [ ]:
display(df_config)

SOURCE_DB_NAME,SOURCE_TABLE_NAME,SOURCE_IP,SOURCE_USER_NAME,SOURCE_PASSWORD,S3_BUCKET_PATH,LOAD_DTTM,JOB_ID,TARGET_SCHEMA_NAME,TARGET_TABLE_NAME,TARGET_CATALOGUE_NAME,SOURCE_RECORDS_COUNT,COUNT_VALIDATION,COUNT_VALIDATION_ERROR_DESC,PRECISION_VALIDATION,PRECISION_VALIDATION_ERROR_DESC,DATATYPES_VALIDATION,DATATYPES_VALIDATION_ERROR_DESC,SOURCE_SCHEMA,SOURCE_PORT
mytestdb,customer,mytestdb.chwoam4ia2vw.us-east-1.rds.amazonaws.com,aws_admin,aws_admin,s3://opendoordemo/Postgres_data_from_databricks/mytestdb_customer/mytestdb_customer_2024-05-03_18-46-36.csv,2024-05-03_18-46-36,null,test_s,customer,test_c,38,TRUE,null,null,null,TRUE,null,"{""fields"":[{""metadata"":{""scale"":0},""name"":""customer_id"",""nullable"":true,""type"":""integer""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(50)"",""scale"":0},""name"":""first_name"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(50)"",""scale"":0},""name"":""last_name"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(50)"",""scale"":0},""name"":""email"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(20)"",""scale"":0},""name"":""phone"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(200)"",""scale"":0},""name"":""address"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(200)"",""scale"":0},""name"":""city"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(20)"",""scale"":0},""name"":""state"",""nullable"":true,""type"":""string""},{""metadata"":{""__CHAR_VARCHAR_TYPE_STRING"":""varchar(20)"",""scale"":0},""name"":""postal_code"",""nullable"":true,""type"":""string""}],""type"":""struct""}",null
mytestdb,orders,mytestdb.chwoam4ia2vw.us-east-1.rds.amazonaws.com,aws_admin,aws_admin,s3://opendoordemo/Postgres_data_from_databricks/mytestdb_orders/mytestdb_orders_2024-05-03_18-46-46.csv,2024-05-03_18-46-46,null,test_s,orders,test_c,38,TRUE,null,null,null,TRUE,null,"{""fields"":[{""metadata"":{""scale"":0},""name"":""orderid"",""nullable"":true,""type"":""integer""},{""metadata"":{""scale"":0},""name"":""customerid"",""nullable"":true,""type"":""integer""},{""metadata"":{""scale"":0},""name"":""orderdate"",""nullable"":true,""type"":""date""},{""metadata"":{""scale"":2},""name"":""totalamount"",""nullable"":true,""type"":""decimal(10,2)""}],""type"":""struct""}",null


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
import json

def load_csv_from_s3(s3_path, schema=None):
    """
    Load CSV data from an S3 bucket into a PySpark DataFrame.
    
    Args:
    - spark: SparkSession object
    - s3_path (str): Path to the CSV file in the S3 bucket
    - schema (StructType, optional): Schema to apply to the DataFrame
    
    Returns:
    - DataFrame: PySpark DataFrame containing the loaded CSV data
    """
    # Define DataFrameReader options
    options = {"header": "true", "inferSchema": "true"}
    print('original schema ',type(schema))
    schema_json = json.loads(schema)
    struct_schema = StructType.fromJson(schema_json)
    print(struct_schema)
    # If schema is provided, apply it
    if schema:
        options["schema"] = struct_schema
    
    # Read CSV data into DataFrame
    df = spark.read.csv(s3_path, **options)
    
    return df

In [ ]:
def check_if_counts_match(source_count, target_count):
    counts_matched = 'FALSE'
    if source_count == target_count:
        print("counts are matching")
        counts_matched = 'TRUE'
    return counts_matched

In [ ]:
def check_if_schemas_match(source_schema, target_schema):
    schema_matched = 'FALSE'
    if source_schema == target_schema:
        print('schema match')
        schema_matched = 'TRUE'
    else:
        print('schema does not match')
    return schema_matched

In [ ]:
import json

for row in df_config.collect():
    print('loop start')
    data_df = load_csv_from_s3(row['S3_BUCKET_PATH'], row['SOURCE_SCHEMA'])

    print(data_df.schema)
    records_count = data_df.count()
    print('target records count',records_count)
    print('source records count',int(row['SOURCE_RECORDS_COUNT']))

    counts_matched = check_if_counts_match(int(row['SOURCE_RECORDS_COUNT']),records_count)
   
    table_name = f"{row['TARGET_CATALOGUE_NAME']}.{row['TARGET_SCHEMA_NAME']}.{row['TARGET_TABLE_NAME']}"
    delete_table_sql = f"""
                        drop table {table_name}
                        """
    spark.sql(delete_table_sql)
    print(table_name)
    data_df.write.format("delta").saveAsTable(table_name)

    source_schema_json = json.loads(row['SOURCE_SCHEMA'])
    
    print('source_schema', source_schema_json)
    print('source schema type ', type(source_schema_json))
    print('------------------------')
    print('target schema ', data_df.schema.json())
    print('target schema type ', type(data_df.schema.json()))
    
    schema_matched = check_if_schemas_match(source_schema_json,json.loads(data_df.schema.json()))
    

    update_config_sql = f"""
                        UPDATE risam_demo1.opendoor.source_to_s3_dataload_config
                        SET COUNT_VALIDATION = '{counts_matched}', DATATYPES_VALIDATION='{schema_matched}'
                        WHERE SOURCE_TABLE_NAME = '{row['SOURCE_TABLE_NAME']}' AND SOURCE_DB_NAME = '{row['SOURCE_DB_NAME']}'
                        """
    spark.sql(update_config_sql)

loop start
original schema  <class 'str'>
StructType([StructField('customer_id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('phone', StringType(), True), StructField('address', StringType(), True), StructField('city', StringType(), True), StructField('state', StringType(), True), StructField('postal_code', StringType(), True)])
StructType([StructField('customer_id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('phone', StringType(), True), StructField('address', StringType(), True), StructField('city', StringType(), True), StructField('state', StringType(), True), StructField('postal_code', StringType(), True)])
target records count 38
source records count 38
counts are matching
test_c.test_s.customer
source_schema {'fields': [{'metadata': 